# Mini Transformer
**Cours NLP - CY Tech**

**Nom : EL ABED Khaled**

# TP1 - Du texte brut au premier modèle bigramme

### Librairies

In [1]:
import urllib.request
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

### Étape 1 : télécharger et inspecter le corpus

In [2]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = urllib.request.urlopen(url).read().decode("utf-8")
print("Longueur :", len(text))
print(text[:300])

Longueur : 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


La longueur du corpus: Le corpus contient environ 1,115,394 caractères.

Les retours à la ligne: On voit des retours à la ligne qui séparent les noms de personne de leurs répliques et qui séparent aussi les répliques entre elles.

La ponctuation: On trouve tous les ponctuation classique comme des virgules, des points, des points d'interrogation et etc.

Les majuscules et minuscules: Le texte contient une melange des majuscules et des minuscules. Il sont traitées comme des caractères différents dans le vocabulaire.

Q1: Quelle est ici l'unité d'un token?

Ici l'unité d'un token est un caractère donc une lettre, un espace, un signe de ponctuation ou un retour à la ligne et pas un mot.
Q2: Pourquoi préciser encoding="utf-8"?

On précise utf-8 pour être sûr que le fichier soit lu de la même façon sur toutes les machines. Sinon certains caractères spéciaux pourraient être mal décodés.
Q3: Le fichier estil déjà séparé en exemples?

Non, le fichier est une seule longue chaîne de caractères continue. Il faut donc que nous construisions nous-mêmes les exemples. C'est-à-dire les couples contexte et cible, à partir de ce texte.


### Étape 2 : passer des caractères aux entiers

In [3]:
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join(itos[i] for i in l)

print("Taille du vocabulaire :", vocab_size)
message = "Hello"
print(encode(message), decode(encode(message)))
assert decode(encode(message)) == message

Taille du vocabulaire : 65
[20, 43, 50, 50, 53] Hello


Q1: Pourquoi trier chars?
On trie les chars pour garantir un ordre stable et reproductible. Si on ne trie pas les chars, l'ordre des caractères pourrait changer d'une exécution à l'autre ce qui donnerait des identifiants différents pour les mêmes caractères.
Q2: Que garantit le test decode(encode(message)) == message ?

Ce test garantit que encode et decode sont bien les fonctions inverses l'une de l'autre. Donc aucune information n'est perdue lors de la conversion du texte en entiers puis à nouveau en texte.

### Étape 3 : fabriquer les prédictions attendues

In [4]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size + 1]
for t in range(block_size):
    print(f"contexte {x[:t+1].tolist()} -> cible {y[t].item()}")

contexte [18] -> cible 47
contexte [18, 47] -> cible 56
contexte [18, 47, 56] -> cible 57
contexte [18, 47, 56, 57] -> cible 58
contexte [18, 47, 56, 57, 58] -> cible 1
contexte [18, 47, 56, 57, 58, 1] -> cible 15
contexte [18, 47, 56, 57, 58, 1, 15] -> cible 47
contexte [18, 47, 56, 57, 58, 1, 15, 47] -> cible 58


Q1: Combien de prédictions produit ce bloc?
Ce bloc produit 8 prédictions. Une pour chaque position t dans range block_size.
Q2: Pourquoi ne pas mélanger train et validation?

On ne mélange pas train et validation car le texte est séquentiel. Mélanger casserait l'ordre du texte et pourrait faire fuir des informations entre les deux ensembles.

Q3: Pour t = 3, quels sont le contexte et la cible?
Pour t = 3, le contexte est x[:4] donc c'est les 4 premiers caractères de la fenêtre et la cible est y[3], le caractère qui suit directement ce contexte

### Étape 4 : échantillonner un mini-batch

In [5]:
batch_size = 4

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x, y

xb, yb = get_batch("train")
print(xb.shape, yb.shape)

torch.Size([4, 8]) torch.Size([4, 8])


Q1: Pourquoi retirer block_size à la borne de tirage?
On retire block_size à la borne de tirage pour éviter qu'une position de départ tirée aléatoirement soit trop proche de la fin du texte.

Q2: Que changerait B = 32?
Avec B = 32, on traiterait 32 séquences indépendantes en parallèle au lieu de 4, ce qui rendrait l'entraînement plus rapide et les gradients plus stables mais demanderait beaucoup plus de mémoire.
Q3: Que changerait T = 64?

Avec T = 64, chaque séquence contiendrait 64 caractères de contexte au lieu de 8. En effet, ca permettrait au modèle de voir plus de contexte mais rendrait chaque calcul plus coûteux.

### Étape 5 : une table produit les logits

In [6]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)          # (B, T, C)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, idx_next], dim=1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape, loss.item())

torch.Size([4, 8, 65]) 5.036386013031006


Q1: Pourquoi C × C ici?

C'est C × C car ce modèle bigramme ne regarde qu'un seul caractère précédent. Il faut donc, pour chacun des C caractères possibles en entrée, un score pour chacun des C caractères possibles en sortie.

Q2: Où retrouvera-t-on demb et la couche lm_head dans le TP 4?
Dans le TP 4, la table d'embedding ne produira plus directement des logits mais un vecteur de dimension d_emb par token. Ce vecteur passera ensuite par les blocs d'attention et de MLP puis la couche lm_head convertira le résultat final en logits sur le vocabulaire.

### Étape 6 : générer, puis apprendre

In [7]:
idx = torch.zeros((1, 1), dtype=torch.long)
print("AVANT entraînement :")
print(decode(m.generate(idx, 200)[0].tolist()))

AVANT entraînement :

lfJeukRuaRJKXAYtXzfJ:HEPiu--sDioi;ILCo3pHNTmDwJsfheKRxZCFs
lZJ XQc?:s:HEzEnXalEPklcPU cL'DpdLCafBheHd:.?nA!LgjgL&jcM-sZW!HFsWt'rv.x;z&hPyd'XHOxnAcj.erRjcLxxVFSrOpW.oeGJ Ai$yi&!mRfeopx,f.e..iBgxVrl-VTh


### Étape 7 : entraîner (100 puis 1000 étapes)

In [8]:
def train_bigram(model, steps, lr=1e-2):
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    for step in range(steps):
        xb, yb = get_batch("train")
        _, loss = model(xb, yb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    return loss.item()

m = BigramLanguageModel(vocab_size)
print("Loss après 100 étapes :", train_bigram(m, 100))
print(decode(m.generate(idx, 200)[0].tolist()))

print("Loss après 1000 étapes :", train_bigram(m, 900))
print(decode(m.generate(idx, 200)[0].tolist()))

Loss après 100 étapes : 4.077887535095215

vI;SIuk!lIf.LO!lerDX:Ci$ZBlgdzJvmyYSlBJvjxZcDxXIRCiD$3$K$R;-vrWgAfZQ ,R:kW.MMMuqVA:kxVbX'C:k!
ALNX,wJGSL-,RAwJtMPlNP!lxxXY:p$YazjWnic'LhuW-tnJlzfu;WanWQHZ
'GyDZTPyBB;gtwV$dHr-I'S!x-cxXYUOMNjS!hA gPGH 
Loss après 1000 étapes : 2.4347026348114014


Whilkestyr:
AQGey Br

AL-FleNI y honellonghefre,
SClo h we d:O:

R:
Vs!fer
LUORbjby g?THpanoandiseron; em, lYerewilldo$Y:R:
SIfGUKERUpt send ayoFleyont llMIRoY ceds ag!PESI whe s wr,
d, e bioa&XENTom


1: Pourquoi ne garde-t-on que le dernier pas de temps pendant la génération?
On ne garde que le dernier pas de temps parce que pendant la génération, on ajoute un seul nouveau caractère à la fois. Seule la prédiction pour la position la plus récente est utile pour choisir ce caractère suivant.

2: Pourquoi aplatit-on B × T pendant la loss?
On aplatit B × T car cross_entropy attend une prédiction par exemple pas une prédiction par séquence. Aplatir permet de traiter chaque position de chaque séquence comme un exemple de classification indépendant.

3: Comparez la loss et la structure du texte avant/après 100 étapes, puis avec 1000 étapes
à 100 étapes la loss diminue de 4.57 et à 3.48 à 1 000 étapes. Ca montre que le modèle apprend progressivement des régularités statistiques entre caractères. Cependant, le texte généré reste très proche du bruit dans les deux cas, sans mots reconnaissables ni structure grammaticale. En effet, ca s'explique par la limite structurelle du modèle bigramme. Il ne regarde qu'un seul caractère précédent et il ne peut donc jamais capturer autre chose que des statistiques très locales quel que soit le nombre d'étapes d'entraînement.

# TP2 - Construire l'auto-attention pas à pas

### Étape 1 : agréger par produit matriciel

In [9]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / a.sum(1, keepdim=True)
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b
print("a =", a, "\nb =", b, "\nc =", c)

a = tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]]) 
b = tensor([[2., 7.],
        [6., 4.],
        [6., 5.]]) 
c = tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


Q1: Sans Python, calculez c2. Pourquoi la somme de chaque ligne de a vaut-elle 1?
c2 est la moyenne des trois lignes de b. Donc (b0 + b1 + b2) / 3, car la ligne 2 de a vaut [1/3, 1/3, 1/3]. Chaque ligne de a somme à 1 car elle a été normalisée en divisant par la somme de ses valeurs. En effet, ca garantit que c est bien une moyenne pondérée et pas une simple somme.

### Étape 2 : moyenner uniquement le passé (version boucle)

In [10]:
B, T, C_ = 4, 8, 2
x = torch.randn(B, T, C_)
xbow = torch.zeros(B, T, C_)
for b_ in range(B):
    for t in range(T):
        xprev = x[b_, :t + 1]            # (t+1, C)
        xbow[b_, t] = xprev.mean(dim=0)
print(xprev.shape)

torch.Size([8, 2])


Q1: Donnez les formes de xprev pour (t=0), (t=3) et (t=7). moyenne porte-t-elle sur dim=0?
xprev a pour forme (1, C) à t=0, (4, C) à t=3, et (8, C) à t=7. Parce que xprev contient toujours t+1 vecteurs. On moyenne sur dim=0 car c'est la dimension des positions. On veut réduire le nombre de positions à 1 tout en gardant les C features intactes.

### Étapes 3 et 4 : vectoriser (matrice, puis masque + softmax)

In [11]:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x
print("Étape 3 identique à la boucle :", torch.allclose(xbow, xbow2))

tril = torch.tril(torch.ones(T, T))
wei = torch.zeros(T, T).masked_fill(tril == 0, float("-inf"))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
print("Étape 4 identique :", torch.allclose(xbow, xbow3))

Étape 3 identique à la boucle : True
Étape 4 identique : True


Q1: Pourquoi wei @ x fonctionne-t-il avec une matrice (T , T) et un tenseur (B, T, C)?
wei @ x fonctionne car PyTorch applique automatiquement la même matrice (T,T) à chacun des B batches grâce au broadcasting.
Q2: Que devient Softmax aux emplacements contenant −∞?
Aux emplacements contenant −∞, Softmax donne exactement 0. Car exp(−∞) = 0. Ces positions futures reçoivent donc un poids nul.

### Étapes 5 et 6 : poids d'attention (Q, K, V) et softmax stabilisé

In [12]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k, q, v = key(x), query(x), value(x)
wei = q @ k.transpose(-2, -1) * head_size ** -0.5     # mise à l'échelle (étape 6)
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float("-inf"))
wei = F.softmax(wei, dim=-1)
out = wei @ v

print("Somme des lignes :", wei[0].sum(-1))
print(wei[0])
print("Forme de out :", out.shape)

Somme des lignes : tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
       grad_fn=<SumBackward1>)
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3966, 0.6034, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3069, 0.2892, 0.4039, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3233, 0.2175, 0.2443, 0.2149, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1479, 0.2034, 0.1663, 0.1455, 0.3369, 0.0000, 0.0000, 0.0000],
        [0.1259, 0.2490, 0.1324, 0.1062, 0.3141, 0.0724, 0.0000, 0.0000],
        [0.1598, 0.1990, 0.1140, 0.1125, 0.1418, 0.1669, 0.1061, 0.0000],
        [0.0845, 0.1197, 0.1078, 0.1537, 0.1086, 0.1146, 0.1558, 0.1553]],
       grad_fn=<SelectBackward0>)
Forme de out : torch.Size([4, 8, 16])


Questions d’interprétation
Q1: Quelle ligne de wei[0] correspond au dernier token?
La dernière ligne de wei[0] (ligne T-1) correspond au dernier token parce que chaque ligne i contient les poids d'attention du token i vers tous les tokens précédents.
Q2: Pourquoi la sortie mélanget-elle (v), et non directement (k)?
La sortie mélange v et non k car k sert uniquement à mesurer la pertinence par rapport à la query. Alors que v contient l'information réellement transmise une fois cette pertinence calculée

Vérifiez la somme des lignes de wei, l'absence de poids vers le futur et la forme finale. Expliquez le rôle distinct de Q, K et V.
La somme de chaque ligne de wei vaut bien 1.0 pour les 8 positions. Ce qui confirme que Softmax produit une vraie distribution de probabilité sur chaque ligne. La matrice wei[0] montre bien des zéros au-dessus de la diagonale (par exemple la ligne 0 vaut [1., 0., 0., 0., 0., 0., 0., 0.]). En effet, ca confirme que le masque causal empêche tout accès au futur. La forme finale de out est torch. Size([4, 8, 16]) soit (B, T, head_size) comme attendu.

Q, K et V ont des rôles distincts: Q représente ce que le token cherche, K représente ce qu'un token peut offrir pour être trouvé et V représente l'information réellement transmise une fois qu'un token a été jugé pertinent par le produit Q·K.

# TP3 - Stabiliser puis transformer chaque token

### Étape 1 : coder LayerNorm

In [13]:
class LayerNorm1d:
    def __init__(self, dim, eps=1e-5):
        self.eps = eps
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

    def __call__(self, x):
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, keepdim=True, unbiased=False)
        xhat = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * xhat + self.beta

ln = LayerNorm1d(100)
x = torch.randn(32, 100)
y = ln(x)
print(y.shape)

torch.Size([32, 100])


Sur quel axe normaliser ? À quoi servent ε, γ et β ?
Q1: On normalise sur l'axe des features (dim=-1). Pour chaque token indépendamment, à travers ses propres composantes. ε évite une division par zéro si la variance est nulle. γ et β sont deux paramètres appris qui permettent au modèle de réajuster l'échelle et le décalage après la normalisation.


### Étape 2 : choisir l'axe de normalisation

In [14]:
# Prédis d'abord quelle ligne donnera environ (0, 1), puis vérifie :
print("x[0, :] ->", y[0, :].mean().item(), y[0, :].std().item())
print("x[:, 0] ->", y[:, 0].mean().item(), y[:, 0].std().item())

x[0, :] -> 2.8610228852699038e-08 1.0050320625305176
x[:, 0] -> 0.028664082288742065 1.0553985834121704


Questions d’interprétation
Q1: Prédisez quelle ligne donnera environ (0, 1) avant d’exécuter le code.
Je pense que c'est x[0, :] qui donnera environ (0, 1) parce que LayerNorm normalise chaque token à travers ses propres features. Pas une feature donnée à travers tout le batch. x[:, 0] n'a donc aucune raison d'avoir une moyenne de 0 ou un écart type de 1.

### Étape 3 : réseau feed-forward

In [15]:
n_embd = 32
ffwd = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.ReLU(), nn.Linear(4 * n_embd, n_embd))
x = torch.randn(4, 8, n_embd)
h = F.relu(ffwd[0](x))
print("Proportion d'activations positives :", (h > 0).float().mean().item())
delta = ffwd(x)
print(delta.shape)

Proportion d'activations positives : 0.49609375
torch.Size([4, 8, 32])


Q1: Quelle proportion d’activations est positive ? Pourquoi revenir à C?
Quelle proportion d’act Après ReLU, environ la moitié des activations sont positives et l'autre moitié sont ramenées à 0, car les valeurs avant ReLU sont réparties de façon à peu près symétrique autour de 0 avec des poids initialisés aléatoirement. On revient à C pour que le résultat delta ait la même forme que le vecteur d'embedding initial, ce qui permet de l'ajouter à ce vecteur via la connexion résiduelle.

### Étapes 4 et 5 : ajouter le résiduel

In [16]:
ln = nn.LayerNorm(n_embd)
x_out = x + ffwd(ln(x))
print(x_out.shape)

torch.Size([4, 8, 32])


Q1: Quelle dimension LayerNorm conserve-t-elle?
LayerNorm conserve la dimension d'embedding. La forme du tenseur en entrée et en sortie reste identique. LayerNorm ne fait que recentrer et redimensionner les valeurs à l'intérieur de chaque token.
Q2: Le MLP mélange-t-il les tokens?
Non, le MLP ne mélange pas les tokens. La même transformation est appliquée indépendamment à chaque token. Sans aucune interaction entre les positions. Seul le mécanisme d'attention mélange l'information entre les tokens.
Q3: À quoi sert la connexion résiduelle?
La connexion résiduelle permet d'ajouter la correction calculée (par l'attention ou le MLP) au vecteur d'origine plutôt que de le remplacer. Cela préserve l'information initiale à travers les couches et facilite la propagation du gradient pendant l'entraînement. En effet, sa rendre les réseaux profonds plus faciles à entraîner.

# TP4 - Assembler et entraîner le modèle complet

### Étape 0 : préparer le notebook (hyperparamètres)

In [17]:
batch_size = 16
block_size = 32
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
learning_rate = 1e-3
max_iters = 3000
eval_interval = 500
eval_iters = 100
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device :", device)

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

Device : cpu


Q1: Calculez dh = nemb/nhead.
d_h = n_embd / n_head = 64 / 4 = 16.
Q2: Combien de tokens contient un batch?
Un batch contient batch_size × block_size = 16 × 32 = 512 tokens.
Q3: Sur CPU, quelle variable réduiriez-vous d’abord pour tester?
Je réduirais d'abord max_iters juste pour vérifier rapidement que le pipeline fonctionne sans erreur avant de lancer un entraînement complet plus long.

### Étape 1 : une tête d'attention

In [18]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        wei = q @ k.transpose(-2, -1) * k.size(-1) ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        return wei @ v

Q1: Pourquoi utiliser k.size(-1) plutôt que (C) pour le facteur d’échelle?
On utilise -1 car dans une attention multi-têtes, les vecteurs Key ont pour dimension head_size qui est plus petite que C. Le facteur d'échelle doit correspondre à la vraie dimension des vecteurs comparés et pas à la dimension d'embedding complète.
Q2: Pourquoi tronquer tril[:T,:T]?
On tronque tril[:T,:T] car ce masque est créé une seule fois à la taille maximale. Mais la séquence traitée peut avoir une longueur T plus courte. Il faut donc adapter la taille du masque à la longueur réelle de la séquence en cours.

### Étape 2 : plusieurs têtes

In [19]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

Questions d’interprétation
Q1: Pourquoi concaténer sur la dernière dimension?
On concatène sur la dernière dimension car c'est la dimension des features.
Q2: Quel problème aurait-on si nemb n’était pas divisible par nhead?
Si n_embd n'était pas divisible par n_head, les têtes ne pourraient pas se partager n_embd en portions égales de head_size. La concaténation des sorties ne redonnerait donc pas exactement n_embd dimensions, ce qui empêcherait l'addition résiduelle avec l'entrée d'origine.

### Étape 3 : assembler un bloc

In [20]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.sa = MultiHeadAttention(n_head, n_embd // n_head)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

Questions d’interprétation
Dans quel ordre appliquer LayerNorm, attention/MLP et addition?
On applique d'abord LayerNorm, puis l'attention ou le MLP sur le résultat normalisé. Puis on ajoute ce résultat à l'entrée d'origine non normalisée via la connexion résiduelle.
À quelle opération correspondent « communication » et « calcul »?
Communication correspond à l'attention, car c'est la seule opération qui permet aux tokens d'échanger de l'information entre eux. Calcul correspond au MLP, car il traite chaque token indépendamment sans interaction entre les positions.

### Étapes 4, 5 et 6 : couches du modèle, passage avant + loss, génération

In [21]:
class MiniTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)                                  # (B, T, n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))  # (T, n_embd)
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                                                   # (B, T, vocab_size)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]          # contexte limité
            logits, _ = self(idx_cond)
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

model = MiniTransformer().to(device)
print(sum(p.numel() for p in model.parameters()) / 1e6, "M paramètres")

0.209729 M paramètres


Étape 4

Q1: Donnez les dimensions des trois matrices d’embedding/unembedding.
token_embedding_table a pour dimensions (vocab_size, n_embd), position_embedding_table a pour dimensions (block_size, n_embd), et lm_head a pour dimensions (n_embd, vocab_size).

Q2: Pourquoi lm_head n’est-il plus confondu avec l’embedding ?
Contrairement au TP1, il y a maintenant de vrais calculs (attention et MLP) entre l'embedding et la prédiction finale. Le token embedding donne une représentation initiale du caractère tandis que lm_head convertit la représentation finale, déjà contextualisée, en logits sur le vocabulaire. Ce sont deux transformations apprises séparément, avec des rôles différents.
Étape 5

Q3: Quelle règle de broadcasting permet l’addition des embeddings?
tok_emb a pour forme (B, T, n_embd) et pos_emb a pour forme (T, n_embd). Le broadcasting de PyTorch aligne les dimensions à partir de la droite : les deux dernières dimensions correspondent déjà, et la dimension B manquante dans pos_emb est automatiquement répétée pour chaque séquence du batch.

Q4: Pourquoi ne pas aplatir les logits lors de la génération?
Aplatir les logits n'est utile que pour calculer la loss avec cross_entropy, qui attend une forme (B*T, vocab_size). Pendant la génération, il n'y a pas de cible ni de loss à calculer : on a seulement besoin des logits de la dernière position, donc garder la forme (B, T, vocab_size) et ne sélectionner que le dernier pas de temps est plus simple.
Étape 6

Q5: Pourquoi @torch.no_grad() ?
@torch.no_grad() désactive le calcul des gradients. Cela économise de la mémoire et du calcul.
Q6: Pourquoi échantillonner avec multinomial plutôt que prendre systématiquement argmax ?

### Étapes 7 et 8 : évaluer puis entraîner

In [22]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
for it in range(max_iters):
    if it % eval_interval == 0 or it == max_iters - 1:
        l = estimate_loss()
        print(f"étape {it} : train {l['train']:.4f} | val {l['val']:.4f}")
    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

étape 0 : train 4.3264 | val 4.3298
étape 500 : train 2.2720 | val 2.2771
étape 1000 : train 2.0637 | val 2.1031
étape 1500 : train 1.9302 | val 2.0157
étape 2000 : train 1.8539 | val 1.9486
étape 2500 : train 1.7846 | val 1.9170
étape 2999 : train 1.7393 | val 1.8903


Q1: Pourquoi appeler model.eval() puis model.train()?
model.eval() met le modèle en mode évaluation. C'est important pour des couches comme Dropout qui se comportent différemment à l'entraînement et à l'inférence. model.train() remet ensuite le modèle en mode entraînement pour que les étapes suivantes se comportent normalement.
Q2: Si la loss train baisse mais que la loss validation remonte, que se passe-t-il?
Le modèle sur-apprend: il mémorise de plus en plus les données d'entraînement au lieu d'apprendre des régularités généralisables. En effet, ca améliore son score sur le train mais dégrade sa performance sur des données non vues.

### Étape 9 : générer et expérimenter

In [23]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, 500)[0].tolist()))


Shorring newell. 
'TMANUS:
Sees, ums antendly.

AUULET:
I gall by rielal borish rous! begents fastest are ucloste,
And All:
Ayal! Sice in oute, brettus to rest of tere on this;'
Which at tor this rippes with be' tho killy, manry carklile bodevy deave,
But Romeo, if there gon, more are ayver!

OF OF Sorcuredes
Lever thed at freach; conten so thime eary all him
Ough me; yobre thee brother with can their doost not.

GLORESS, are gones,
This glace to me ses your forther, veawich to ecaurs:
To by vic


Bilan des quatre TP
Le token embedding et le position embedding créent le contexte initial.

L'attention (Head, MultiHeadAttention) transforme chaque token en mélangeant l'information des tokens précédents.

Le MLP (FeedForward) transforme chaque token indépendamment. Le lm_head produit les logits finaux sur le vocabulaire.

La loss agit sur l'ensemble du modèle.

La rétropropagation fait remonter le gradient à travers lm_head, les blocs, jusqu'aux embeddings, ajustant tous les paramètres pour réduire l'erreur de prédiction.
